# GIADA Task 17f — supporto eventi e conferma causale
Pilot sul nativo, congelamento degli stimoli, conferma con seed separati. Consulta `experiments/task17f_event_supported_confirmation_preregistration.md`. CPU; nessun dataset esterno. La GPU non è necessaria. L'output distingue supporto insufficiente, riferimento non affidabile e fallimento del candidato.

In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys, uuid
from IPython.display import Javascript, display
WORK=Path('/kaggle/working')/f'giada_task17f_{uuid.uuid4().hex[:8]}'
REPO=WORK/'giada'; TEACHER=WORK/'neuron_as_deep_net'; WORK.mkdir(parents=True)
SOURCE_REVISION=os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',SOURCE_REVISION],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
for filename in ('scripts/run_roadmap_task17f.py','experiments/task17f_event_supported_confirmation_preregistration.md','notebooks/17f_roadmap_event_supported_confirmation.ipynb'):
 assert (REPO/filename).is_file(),f'Revisione incompleta: {filename}'
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})

## Ambiente e compilazione
La versione installata viene verificata in un sottoprocesso, così il kernel del notebook non carica librerie native. I log completi restano su disco.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','scipy','pandas','matplotlib','pyyaml','h5py','pyarrow'],check=True)
subprocess.run([sys.executable,'-c',"import neuron; assert neuron.__version__.split('+',1)[0]=='8.2.7',neuron.__version__"],check=True)
os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
nrnivmodl=shutil.which('nrnivmodl'); assert nrnivmodl and shutil.which('gcc'),'Compilatore NEURON/gcc assente'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
with (WORK/'teacher_compile.log').open('w') as log:
 compiled=subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,stdout=log,stderr=subprocess.STDOUT)
if compiled.returncode: raise RuntimeError((WORK/'teacher_compile.log').read_text()[-4000:])
assert list(simulation_dir.rglob('libnrnmech.so')),'Meccanismi canonici non compilati'
print({'NEURON':'8.2.7','canonical_mechanisms':'ready','hardware':'CPU'})

## Pilot e conferma
Fino a 42 episodi pilot; soltanto se il supporto è completo si congela il protocollo e si aprono i seed nuovi. La conferma comprende 144 episodi e una replica attiva. Report, input e tracce dense sono salvati progressivamente; nessun array lungo viene stampato.

In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/f'giada_task17f_event_confirmation_{REVISION[:7]}_{uuid.uuid4().hex[:8]}'
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_roadmap_task17f.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--output',str(OUTPUT)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').is_file() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'decision':'SUPERVISOR_INTERRUPTED','error':f'exit={completed.returncode}'}
display({k:report.get(k) for k in ('valid','decision','gate_c_authorized','task18_authorized','gates','support','pilot_trial_count','completed_trial_count','comparison_count','error')})
print({'output':str(OUTPUT),'returncode':completed.returncode})

## Archivio automatico e download opzionale
Lo ZIP viene sempre creato per il recupero via MCP. In un run interattivo la cella avvia anche il download nel browser con il metodo già verificato. `GIADA_MANUAL_DOWNLOAD=0` disabilita solo il popup, non l'archiviazione.

In [ ]:
EXPORT=WORK/'export'; EXPORT.mkdir(exist_ok=False)
for name in ('final_report.json','failure_report.json','process_status.json','process.log','last_phase.json','run_contract.json','code_provenance.json','protocol_freeze.json','pilot_progress.json','confirmation_support.json','native_repeat.json','paired_metrics.json','paired_effects.json','runtime_seconds.json'):
 source=OUTPUT/name
 if source.is_file():shutil.copy2(source,EXPORT/name)
for name in ('pilot_traces','confirmation_traces'):
 if (OUTPUT/name).is_dir():shutil.copytree(OUTPUT/name,EXPORT/name)
shutil.copy2(REPO/'experiments/task17f_event_supported_confirmation_preregistration.md',EXPORT/'preregistration.md')
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',EXPORT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','1')!='0':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})